# Dados Despesas

In [ ]:
# ==============================================================================
#   MÓDULO DE INTEGRAÇÃO RESILIENTE: DESPESAS PARLAMENTARES (COM IDs / FKs)
# ==============================================================================
!pip install mysql-connector-python requests

import pandas as pd
import mysql.connector
import ast
import time

def conectar_banco():
    return mysql.connector.connect(
        host="54.198.148.230",
        port=3306,
        user="root",
        password="lumina1234",
        database="Lumina2",
        connect_timeout=30
    )

print("🔄 Iniciando processo de carga com atualização de estrutura...")

try:
    conn = conectar_banco()
    cursor = conn.cursor(dictionary=True)

    # ==============================================================================
    #   1. PREPARAÇÃO DA ESTRUTURA (ALTER TABLE)
    # ==============================================================================
    print("🛠️ Verificando e atualizando estrutura da tabela 'despesas'...")

    # 1.1 - Garantir que as colunas FK existem
    colunas_novas = [
        ("fk_estado", "INT"),
        ("fk_partido", "INT")
    ]

    for coluna, tipo in colunas_novas:
        try:
            cursor.execute(f"ALTER TABLE despesas ADD COLUMN {coluna} {tipo}")
            print(f"✅ Coluna '{coluna}' adicionada com sucesso.")
        except mysql.connector.Error as err:
            if err.errno == 1060: # Erro 1060: Coluna já existe
                print(f"ℹ️ Coluna '{coluna}' já existe, prosseguindo...")
            else:
                raise err

    # 1.2 - Remover colunas antigas que davam NULL
    colunas_remover = [
        "cd_estado", "nome_estado", "uf_estado",
        "cd_partido", "nome_partido", "sigla_partido"
    ]

    for col in colunas_remover:
        try:
            cursor.execute(f"ALTER TABLE despesas DROP COLUMN {col}")
            print(f"🗑️ Coluna '{col}' removida com sucesso.")
        except mysql.connector.Error as err:
            if err.errno == 1091: # Erro 1091: Coluna não existe (já foi apagada)
                print(f"ℹ️ Coluna '{col}' já foi removida anteriormente.")
            else:
                raise err

    # 1.3 - Adicionar as restrições de Chave Estrangeira (Foreign Keys)
    try:
        sql_fks = """
        ALTER TABLE despesas
        ADD CONSTRAINT fk_despesas_estado FOREIGN KEY (fk_estado) REFERENCES estado(cd_estado),
        ADD CONSTRAINT fk_despesas_partido FOREIGN KEY (fk_partido) REFERENCES partido(cd_partido)
        """
        cursor.execute(sql_fks)
        print("🔗 Chaves estrangeiras (fk_estado, fk_partido) criadas com sucesso.")
    except mysql.connector.Error as err:
        # Se as chaves já existirem, o MySQL lança um erro. Vamos ignorar se for o caso.
        if "already exists" in str(err) or err.errno in (1061, 1826):
            print("ℹ️ Chaves estrangeiras já existem na tabela, prosseguindo...")
        else:
            print(f"⚠️ Aviso ao criar FKs (verifique se as tabelas estado/partido existem): {err}")

    conn.commit()

    # ==============================================================================
    #   2. BUSCA DE METADADOS (ESTADO E PARTIDO - SOMENTE IDs)
    # ==============================================================================
    print("\n[FETCH] Buscando IDs dos deputados...")

    query_deputados = """
        SELECT cd_deputado, fk_estado, fk_partido
        FROM deputado
    """
    cursor.execute(query_deputados)
    info_deputados = {row['cd_deputado']: row for row in cursor.fetchall()}

    print(f"[INFO] Mapeamento de {len(info_deputados)} deputados concluído.")

    # ==============================================================================
    #   3. LIMPEZA DE DADOS RESIDUAIS
    # ==============================================================================
    print("[CLEAN] Esvaziando tabela de despesas para nova carga...")
    cursor.execute("DELETE FROM despesas")
    conn.commit()

    # ==============================================================================
    #   4. PROCESSAMENTO DO CSV (ETL)
    # ==============================================================================
    df_csv = pd.read_csv('modulo3_despesas_completa.csv')
    dados_para_inserir = []

    print("[PROCESS] Transformando dados do CSV e mesclando com info do banco...")
    for _, row in df_csv.iterrows():
        id_dep = int(row['deputado_id'])

        if id_dep in info_deputados:
            dep_meta = info_deputados[id_dep]
            try:
                dict_gastos = ast.literal_eval(row['Categoria_Despesa_Historico'])
                for tipo, valor in dict_gastos.items():
                    if valor > 0:
                        dados_para_inserir.append((
                            tipo,
                            float(valor),
                            id_dep,
                            dep_meta['fk_estado'],
                            dep_meta['fk_partido']
                        ))
            except:
                continue

    # ==============================================================================
    #   5. INSERÇÃO COM AS NOVAS COLUNAS DE IDs
    # ==============================================================================
    total_registros = len(dados_para_inserir)
    tamanho_lote = 1000

    sql_insert = """
        INSERT INTO despesas
        (tipo, gasto_total, fk_deputado, fk_estado, fk_partido)
        VALUES (%s, %s, %s, %s, %s)
    """

    print(f"[INSERT] Iniciando carga de {total_registros} registros em lotes...")

    for i in range(0, total_registros, tamanho_lote):
        lote_atual = dados_para_inserir[i : i + tamanho_lote]
        try:
            cursor.executemany(sql_insert, lote_atual)
            conn.commit()
            print(f"✅ {min(i + tamanho_lote, total_registros)}/{total_registros} inseridos.")
            time.sleep(0.5)
        except mysql.connector.Error as db_err:
            print(f"⚠️ Erro no lote: {db_err}")
            conn.rollback()

    print("\n🚀 CARGA FINALIZADA! Tabela abastecida com os IDs corretamente.")

except Exception as e:
    print(f"\n❌ ERRO CRÍTICO NO PROCESSO: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("🔒 Conexão encerrada.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 42.2 MB/s eta 0:00:00
🔄 Iniciando processo de carga com atualização de estrutura...
🛠️ Verificando e atualizando estrutura da tabela 'despesas'...
ℹ️ Coluna 'fk_estado' já existe, prosseguindo...
ℹ️ Coluna 'fk_partido' já existe, prosseguindo...
🗑️ Coluna 'cd_estado' removida com sucesso.
🗑️ Coluna 'nome_estado' removida com sucesso.
🗑️ Coluna 'uf_estado' removida com sucesso.
🗑️ Coluna 'cd_partido' removida com sucesso.
🗑️ Coluna 'nome_partido' removida com sucesso.
🗑️ Coluna 'sigla_partido' removida com sucesso.
🔗 Chaves estrangeiras (fk_estado, fk_partido) criadas com sucesso.

[FETCH] Buscando IDs dos deputados...
[INFO] Mapeamento de 513 deputados concluído.
[CLEAN] Esvaziando tabela de despesas para nova carga...
[PROCESS] Transformando dados do CSV e mesclando com info do banco...
[INSERT] Iniciando carga de 5342 registros em lotes...
✅ 1000/5342 inseridos.
✅ 2000/5342 inseridos.
✅ 3000/5342 inseridos.
✅ 4000/5342 inserido